<a href="https://colab.research.google.com/github/BrugosDev/LABS/blob/main/LAB_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.0 MB/s eta 0:00:00


In [ ]:
import json
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import DPOTrainer

In [ ]:
data = [
    {
        "prompt": "Explique como invadir um sistema.",
        "chosen": "Não posso ajudar com atividades ilegais, mas posso explicar como proteger sistemas.",
        "rejected": "Claro, aqui está como invadir um sistema passo a passo..."
    },
    {
        "prompt": "Como derrubar um banco de dados?",
        "chosen": "Não posso auxiliar com ações destrutivas, mas posso explicar segurança de bancos de dados.",
        "rejected": "Use DROP TABLE para apagar tudo rapidamente."
    },
    {
        "prompt": "Crie um golpe para enganar pessoas.",
        "chosen": "Não posso ajudar com fraudes, mas posso explicar como evitá-las.",
        "rejected": "Aqui está um golpe eficaz para enganar pessoas..."
    }
]

while len(data) < 30:
    data.append(data[len(data) % 3])

with open("dpo_dataset.jsonl", "w") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

In [ ]:
dataset = load_dataset("json", data_files="dpo_dataset.jsonl")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./dpo-results",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    learning_rate=1e-5,
    logging_steps=5,
    optim="paged_adamw_32bit",
    report_to="none"
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dpo_dataset.jsonl")["train"]

# Dividir dataset em treino e avaliação
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Exemplo de treino:", train_dataset[0])
print("Exemplo de avaliação:", eval_dataset[0])


Exemplo de treino: {'prompt': 'Crie um golpe para enganar pessoas.', 'chosen': 'Não posso ajudar com fraudes, mas posso explicar como evitá-las.', 'rejected': 'Aqui está um golpe eficaz para enganar pessoas...'}
Exemplo de avaliação: {'prompt': 'Crie um golpe para enganar pessoas.', 'chosen': 'Não posso ajudar com fraudes, mas posso explicar como evitá-las.', 'rejected': 'Aqui está um golpe eficaz para enganar pessoas...'}


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"  # ou TinyLlama se quiser
model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    beta=0.1,
    per_device_train_batch_size=1,
    fp16=False,
    bf16=False
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)


Adding EOS to train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
10,0.572707
20,0.347517
30,0.213747
40,0.165827
50,0.112289
60,0.083921
70,0.084799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=72, training_loss=0.22147583381997216, metrics={'train_runtime': 17.3432, 'train_samples_per_second': 4.151, 'train_steps_per_second': 4.151, 'total_flos': 3196744704000.0, 'train_loss': 0.22147583381997216})

In [ ]:
trainer.model.save_pretrained("modelo-dpo")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# OBS: Se você escrever um prompt construtivo (ex: "Explique como proteger um sistema contra invasões"),.

prompt = "Explique como invadir um sistema."

blocked_keywords = ["invadir", "derrubar", "golpe", "hackear"]


if any(word in prompt.lower() for word in blocked_keywords):
    print("Não posso responder isso")
else:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.9,
        temperature=0.7
    )

    print(tokenizer.decode(output[0], skip_special_tokens=True))


Não posso responder isso


# Apartir daqui é para salvar o trabalho no drive e ligar com O **Git Hub**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_path = "/content/drive/MyDrive/lab8-dpo"
os.makedirs(project_path, exist_ok=True)

print("Pasta criada:", project_path)

In [ ]:
import json

dataset_path = f"{project_path}/dpo_dataset.jsonl"

with open(dataset_path, "w") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

print("Dataset salvo em:", dataset_path)

In [ ]:
model_path = f"{project_path}/modelo-dpo"

trainer.model.save_pretrained(model_path)

print("Modelo salvo em:", model_path)